# Understanding Planet Formation through Protoplanetary Disks

Planets form in **protoplanetary disks**, which are rotating disks of gas and dust surrounding young stars. These disks are the birthplaces of planets, where microscopic dust grains collide, stick together, and grow into planetesimals that eventually form full-fledged planets.

The structure of a protoplanetary disk is often described by its **surface density** $$\Sigma(R)$$ and **temperature profile** $$T(R)$$ as functions of the distance from the central star, $R$. A commonly used model assumes that the surface density decreases with radius as a power law:

$$
\Sigma(R) = \Sigma_0 \left(\frac{R}{R_0}\right)^{-p},
$$

where $\Sigma_0$ is the reference surface density at radius $R_0$ and $p$ is typically between 0.5 and 1.5.

Similarly, the disk’s temperature often follows:

$$
T(R) = T_0 \left(\frac{R}{R_0}\right)^{-q},
$$

where $T_0$ is the temperature at $R_0\$ and $q$ usually ranges from 0.5 to 0.75.

These profiles determine the **sound speed** $$c_s(R)$$ and **scale height** $$H(R)$$ of the disk:

$$
c_s(R) = \sqrt{\frac{k_B T(R)}{\mu m_H}}, \quad H(R) = \frac{c_s(R)}{\Omega(R)},
$$

where $k_B$ is Boltzmann’s constant, $\mu$ is the mean molecular weight, $m_H$ is the hydrogen mass, and $\Omega(R) = \sqrt{GM_\ast/R^3}$ is the Keplerian angular velocity around a star of mass $M_\ast$.

In this lesson, we will **use Python code to simulate the evolution of gas in a protoplanetary disk**, explore how density waves propagate, and visualize the structures that can give rise to planetary formation! 

Optionally, we will examine how **spiral arms** can form due to instabilities in the disk.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Constants (scaled units)
G = 1
M_star = 1

# Create particles
N = 200
r = np.random.uniform(0.5, 5, N)
theta = np.random.uniform(0, 2*np.pi, N)

x = r * np.cos(theta)
y = r * np.sin(theta)

# Circular velocity
v = np.sqrt(G * M_star / r)

vx = -v * np.sin(theta)
vy =  v * np.cos(theta)

plt.scatter(x, y, s=5)
plt.gca().set_aspect('equal')
plt.title("Initial Protoplanetary Disk")
plt.show()

# Physics Behind the Protoplanetary Disk Simulation

This Python code simulates a simple **protoplanetary disk**, capturing the key physics of planet formation: orbital motion, gas drag, and particle growth.

---

## Time Step

`dt = 0.01`

- `dt` is the simulation **time step**, controlling how much time passes per iteration.
- Smaller `dt` improves **numerical stability** and accuracy of particle motion.

---

## Particle Merging (Collisions and Coagulation)

Particles closer than a set `threshold` are **merged** to model **dust coagulation**:

- Velocities are averaged:  
  $$
  \vec{v}_\text{merged} = \frac{\vec{v}_i + \vec{v}_j}{2}
  $$
- One particle is removed.
- This simulates the **growth of planetesimals** from smaller dust grains.

---

## Gravitational Acceleration

Each particle feels the **star’s gravity**, producing **Keplerian orbits**:

$$
\vec{a}_\text{gravity} = -\frac{G M_\ast}{r^2} \hat{r}, \quad
a_x = -\frac{G M_\ast x}{r^3}, \ a_y = -\frac{G M_\ast y}{r^3}
$$

---

## Gas Drag

Linear drag slows particles due to the **gaseous disk**:

$$
\vec{a}_\text{drag} = -\gamma \vec{v}
$$

- Causes particles to **lose energy** and **spiral inward** gradually.

---

## Updating Velocities and Positions

- Using **Euler integration**:
  - Velocity updated from acceleration.
  - Position updated from velocity.
- Approximates **continuous motion** in discrete time steps.

---

## Disk Initialization

Particles are randomly placed in a **circular disk**. Velocities are set for **circular Keplerian motion**:

$$
v = \sqrt{\frac{G M_\ast}{r}}, \quad \vec{v} \perp \vec{r}
$$

---

## Animation Loop

- Each iteration advances the system by one time step.
- Periodic plotting shows:
  - Particles orbiting the star.
  - Collisions forming larger clumps.
  - Gradual inward drift from drag.

---

## Constants and Units

- Scaled units simplify the simulation:
  - Star mass $M_\ast = 1$
  - Gravitational constant $G = 1$
- Focuses on **qualitative dynamics** rather than real units.



In [ ]:
dt = 0.01

def merge_particles(x, y, vx, vy, threshold=0.05):
    keep = np.ones(len(x), dtype=bool)
    
    for i in range(len(x)):
        for j in range(i+1, len(x)):
            dx = x[i] - x[j]
            dy = y[i] - y[j]
            dist = np.sqrt(dx**2 + dy**2)
            
            if dist < threshold:
                vx[i] = (vx[i] + vx[j]) / 2
                vy[i] = (vy[i] + vy[j]) / 2
                keep[j] = False
    
    return x[keep], y[keep], vx[keep], vy[keep]

def step(x, y, vx, vy):
    r = np.sqrt(x**2 + y**2)
    
    ax = -G * M_star * x / r**3
    ay = -G * M_star * y / r**3

    drag_coeff = 0.01

    ax -= drag_coeff * vx
    ay -= drag_coeff * vy
    
    vx += ax * dt
    vy += ay * dt
    
    x += vx * dt
    y += vy * dt

    x, y, vx, vy = merge_particles(x, y, vx, vy)
    
    return x, y, vx, vy

# Constants (scaled units)
G = 1
M_star = 1
dt = 0.01

# Initialize disk
def initialize_disk(N=300):
    r = np.random.uniform(0.5, 5, N)
    theta = np.random.uniform(0, 2*np.pi, N)
    
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    
    v = np.sqrt(G * M_star / r)
    vx = -v * np.sin(theta)
    vy =  v * np.cos(theta)
    
    return x, y, vx, vy

x, y, vx, vy = initialize_disk()

# Animate
for i in range(500):
    x, y, vx, vy = step(x, y, vx, vy)
    
    if i % 50 == 0:
        plt.clf()
        plt.scatter(x, y, s=5)
        plt.gca().set_aspect('equal')
        plt.pause(0.01)

plt.show()

# Physics Behind the Spiral Disk Simulation

This version of the simulation builds on the **basic protoplanetary disk code** but introduces **spiral-seeding perturbations** to explore how spiral structures can form.

---

## Core Physics (Same as Before)

- Particles orbit a central star under **gravity**:
  $$
  \vec{a}_\text{gravity} = -\frac{G M_\ast}{r^2} \hat{r}, \quad
  a_x = -\frac{G M_\ast x}{r^3}, \ a_y = -\frac{G M_\ast y}{r^3}
  $$
- **Gas drag** slows particles linearly:
  $$
  \vec{a}_\text{drag} = -\gamma \vec{v}
  $$
- Velocities and positions are updated via **Euler integration**.
- The disk is initialized randomly in a circular pattern with **Keplerian velocities**.

---

## Key Differences From the First Simulation

### 1. No Particle Merging

- Unlike the first code, **particles do not merge** when they get close.
- This allows the **structure of the disk to evolve freely** without forming clumps.
- Focus is on **wave patterns** rather than planetesimal growth.

### 2. Spiral-Seeding Perturbation

```python
vx += 0.2 * np.cos(2 * theta)
vy += 0.2 * np.sin(2 * theta)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

G = 1
M_star = 1
dt = 0.01

def initialize_disk(N=500):
    r = np.random.uniform(0.5, 5, N)
    theta = np.random.uniform(0, 2*np.pi, N)
    
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    
    v = np.sqrt(G * M_star / r)
    vx = -v * np.sin(theta)
    vy =  v * np.cos(theta)
    
    return x, y, vx, vy

def step(x, y, vx, vy, drag=0.001):
    r = np.sqrt(x**2 + y**2)
    
    ax = -G * M_star * x / r**3
    ay = -G * M_star * y / r**3
    
    ax -= drag * vx
    ay -= drag * vy
    
    vx += ax * dt
    vy += ay * dt
    
    x += vx * dt
    y += vy * dt
    
    return x, y, vx, vy

# Initialize
x, y, vx, vy = initialize_disk()

# Add spiral-seeding perturbation (m = 2 mode)
r = np.sqrt(x**2 + y**2)
theta = np.arctan2(y, x)

vx += 0.2 * np.cos(2 * theta)
vy += 0.2 * np.sin(2 * theta)

# Run simulation
for i in range(800):
    x, y, vx, vy = step(x, y, vx, vy)
    
    if i % 100 == 0:
        plt.clf()
        plt.scatter(x, y, s=5)
        plt.gca().set_aspect('equal')
        plt.title(f"Spiral Formation Step {i}")
        plt.pause(0.01)

plt.show()